In [20]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Normalization
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
import numpy as np
import pandas as pd
import matplotlib as plt
import cv2
import os
import glob

In [21]:
categories = ["fall", "grab", "gun", "hit", "kick", 
              "lying_down", "run", "sit", "sneak", 
              "stand", "struggle", "throw", "walk"]

In [ ]:
video_id = 0

for category in categories:
    video_folder = f"D:/Project/CCTV Motion Sensor Project/Videos/{category}"
    video_files = glob.glob(f"{video_folder}/*.mp4")


    for video_path in video_files:
            output_dir = f"frames/{category}"
            os.makedirs(output_dir, exist_ok=True)

            count = 0
            saved = 0 
            frames = cv2.VideoCapture(video_path)

            while True:
                ret, frame = frames.read()
                if not ret:
                    break
                #cv2.imwrite(f"{output_dir}/frame_{count}.jpg", frame)
                count += 1

                if count % 5 == 0:
                    cv2.imwrite(f"{output_dir}/video_{video_id}_frame_{saved}.jpg", frame)
                    saved += 1

            frames.release()
            video_id += 1

In [22]:
encoder = LabelEncoder()
normalizer = Normalization()
frame_path = "D:/Project/CCTV Motion Sensor Project/frames"
data = []
labels = []
score = 0

for category in categories:
    frame_path = f"D:/Project/CCTV Motion Sensor Project/frames/{category}"

    while True:
        file_path = os.path.join(frame_path, f"frame_{score}.jpg")
        x = cv2.imread(file_path)
        if x is None:
            break
        score += 1
        x = cv2.resize(x, (224, 224))
        data.append(x)
        labels.append(category)


data = np.array(data, dtype=np.float32)
normalizer.adapt(data)
data = normalizer(data)

labels_encoded = encoder.fit_transform(labels)
print(data.shape)
print(labels_encoded)


(60, 224, 224, 3)
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 2 2 2 2 2 2]


In [31]:
data = np.array(data, dtype=np.float32)
labels_encoded = encoder.fit_transform(labels)

x_train, x_test, y_train, y_test = train_test_split(
    data, labels_encoded, test_size=0.2, random_state=42, stratify=labels_encoded
)

In [32]:
normalizer = Normalization()
normalizer.adapt(x_train)

x_train = normalizer(x_train)
x_test = normalizer(x_test)

In [33]:
y_train = to_categorical(y_train)
y_test = to_categorical(y_test)

In [39]:
num_classes = len(encoder.classes_)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(224, 224, 3)),
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),

    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),

    tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

In [40]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [41]:
model.fit(x_train, y_train, validation_data= (x_test, y_test), epochs = 10, batch_size = 32)

Epoch 1/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 4s 707ms/step - accuracy: 0.4583 - loss: 1.7244 - val_accuracy: 0.9167 - val_loss: 0.8473
Epoch 2/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 287ms/step - accuracy: 0.8958 - loss: 1.0573 - val_accuracy: 1.0000 - val_loss: 0.0201
Epoch 3/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 283ms/step - accuracy: 1.0000 - loss: 0.0607 - val_accuracy: 1.0000 - val_loss: 0.0097
Epoch 4/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 343ms/step - accuracy: 1.0000 - loss: 0.0191 - val_accuracy: 1.0000 - val_loss: 0.0014
Epoch 5/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 308ms/step - accuracy: 1.0000 - loss: 0.0040 - val_accuracy: 1.0000 - val_loss: 5.2026e-04
Epoch 6/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 271ms/step - accuracy: 1.0000 - loss: 0.0011 - val_accuracy: 1.0000 - val_loss: 1.1734e-04
Epoch 7/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 290ms/step - accuracy: 1.0000 - loss: 4.3923e-04 - val_accuracy: 1.0000 - val_loss: 1.6459e-05
Epoch 8/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 274ms/step - accuracy: 1.0000 - loss: 1.0084e-04 - val_accuracy

In [42]:
loss, acc = model.evaluate(x_test, y_test)
print("Test accuracy:", acc)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 113ms/step - accuracy: 1.0000 - loss: 8.9407e-08
Test accuracy: 1.0
